In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [2]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_comment"],
        df["category"],
        test_size=0.2,
        random_state=42,
        stratify=df["category"],
    )

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    n_estimators = 200
    max_depth = 15

    # Initialize and train the model
    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)

    # Step 5: Make predictions and evaluate metrics
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    # Display clean results summary in the console
    run_name = f"TFIDF_Trigrams_max_features_{max_features}"
    print("=" * 60)
    print(f"RUN: {run_name}")
    print(f"Configuration: Trigrams (1, 3), max_features={max_features}")
    print("-" * 60)
    print(f"Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("=" * 60 + "\n")

    # Generate and save confusion matrix with a unique filename
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")

    # Unique string prevents filename collisions
    plot_filename = f"confusion_matrix_tfidf_trigrams_{max_features}.png"
    plt.savefig(plot_filename)
    plt.close()

    # Returns results to accumulate for sorting/comparing
    return {"max_features": max_features, "accuracy": accuracy}


# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

# List to store summary statistics
results_log = []

for max_features in max_features_values:
    result = run_experiment_tfidf_max_features(max_features)
    results_log.append(result)

# Print a final benchmark table comparing everything
print("\n##### SUMMARY BENCHMARK #####")
print(f"{'Max Features':<15}{'Accuracy':<10}")
print("-" * 25)
for res in results_log:
    print(f"{res['max_features']:<15}{res['accuracy']:.4f}")


RUN: TFIDF_Trigrams_max_features_1000
Configuration: Trigrams (1, 3), max_features=1000
------------------------------------------------------------
Accuracy: 0.6618

Classification Report:
              precision    recall  f1-score   support

          -1       0.88      0.13      0.23      1650
           0       0.62      0.89      0.73      2529
           1       0.69      0.75      0.72      3154

    accuracy                           0.66      7333
   macro avg       0.73      0.59      0.56      7333
weighted avg       0.71      0.66      0.61      7333


RUN: TFIDF_Trigrams_max_features_2000
Configuration: Trigrams (1, 3), max_features=2000
------------------------------------------------------------
Accuracy: 0.6589

Classification Report:
              precision    recall  f1-score   support

          -1       0.90      0.08      0.14      1650
           0       0.64      0.88      0.74      2529
           1       0.66      0.79      0.72      3154

    accuracy        